# **Jio CareOps Network Issue Analyzer | Student Starter**

## **Your Engineering Handover**

You have joined the Jio Customer Care and Network Operations team as a **Junior Python Developer**. The CareOps lead spends hours switching between customer complaint records, network incident reports, recharge failure logs, and service requests to understand which cases need attention. You have been asked to build a reproducible **customer complaint and network issue analysis workflow** in this notebook.

Your deliverable is **not an app or dashboard**. It is a trustworthy set of cleaned data tables, a complaint-level investigation report, and a concise evidence-based handover to Customer Care and Network Operations.

**How to work:** Make your own copy in Google Drive (`File → Save a copy in Drive`), upload the three provided files, complete the ten milestones in order, and run the assessment cell at the end. Write your own implementation in the code cells. Keep every required named output available for the next task and for assessment.

**Input files (only three):** `jio_complaints_master.csv` · `jio_incident_histories.xlsx` · `jio_service_requests.json`  
**Data:** Fictional training records; not actual Jio customer or network operations records.

> **Important:** The assessment at the end checks key output contracts and consistency. Passing it is not proof that every business calculation or written explanation is correct; reviewers may also inspect selected notebooks.

### **Important Note: Guidelines for Using AI Tools**



You are allowed to use AI tools such as ChatGPT to understand concepts, debug errors, and get assistance with **specific code snippets or individual problems**.

However, **using AI to generate the complete project solution in one go is not permitted**.

You are expected to develop the solution milestone by milestone, apply your understanding of Python and Pandas, and independently verify your code and outputs.

**Remember:** The goal of this project is not just to complete the tasks but to develop practical problem-solving, debugging, and data analysis skills that are expected in real-world development roles.

## **Workspace setup · Data intake**
Run the setup cell and upload the **three input files** when requested. Files stay in your current Colab session; if the runtime restarts, upload them again. Keep original filenames unchanged.

[Link to Dataset](https://drive.google.com/file/d/1hgqvHzcqbEDG5e_sjSXjPljwV2_u7noh/view?usp=sharing)

In [1]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import json
import zipfile
DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
REQUIRED_FILES = ['complaints.csv', 'network_issue_reports.xlsx', 'careops_reference.json']
ARCHIVE = 'jio_careops_project_datasets.zip'
if any(not (DATA_DIR/name).exists() for name in REQUIRED_FILES) and (DATA_DIR/ARCHIVE).is_file():
 with zipfile.ZipFile(DATA_DIR/ARCHIVE) as z:
  names=z.namelist()
  if sorted(names)!=sorted(REQUIRED_FILES):
   raise ValueError(f'Dataset ZIP must contain exactly the three expected files: {REQUIRED_FILES}; found {names}')
  z.extractall(DATA_DIR)
if any(not (DATA_DIR/name).exists() for name in REQUIRED_FILES):
 try:
  from google.colab import files
  print('Upload jio_careops_project_datasets.zip (containing CSV, XLSX and JSON).')
  files.upload()
 except ImportError:
  print('Place the dataset ZIP or three extracted files in',DATA_DIR)
 if (DATA_DIR/ARCHIVE).is_file():
  with zipfile.ZipFile(DATA_DIR/ARCHIVE) as z:
   if sorted(z.namelist())!=sorted(REQUIRED_FILES):
    raise ValueError('Unexpected files in dataset archive')
   z.extractall(DATA_DIR)
missing=[name for name in REQUIRED_FILES if not (DATA_DIR/name).exists()]
if missing: raise FileNotFoundError(f'Missing dataset files: {missing}')
print('Verified the three required Jio sources:', ', '.join(REQUIRED_FILES))

Upload jio_careops_project_datasets.zip (containing CSV, XLSX and JSON).


Saving jio_careops_project_datasets.zip to jio_careops_project_datasets.zip
Verified the three required Jio sources: complaints.csv, network_issue_reports.xlsx, careops_reference.json


## Milestone 01 · Establish the complaint baseline

**From the team:** The CareOps lead wants a transparent complaint intake audit before any triage recommendations are trusted.

**Your responsibility:** Load complaints.csv; inspect volume, unique complaint and customer IDs, missing values, and leading raw categories.

**Handover evidence:** Keep `complaints_df`; display shape, sample, duplicates and missing-value counts.

In [2]:
# Milestone 01 — Establish the complaint baseline
# Task 1: Load complaints.csv; inspect volume, unique complaint and customer IDs, missing values, and leading raw categories.
# Task 2: Produce this required evidence: Keep `complaints_df`; display shape, sample, duplicates and missing-value counts.
# Implement your code below and display the requested outputs.




In [3]:
complaints_df=pd.read_csv(DATA_DIR / 'complaints.csv')
print("=== Complaints DataFrame Shape ===")
print(f"Shape: {complaints_df.shape} (Rows: {complaints_df.shape[0]}, Columns: {complaints_df.shape[1]})\n")

print("=== DataFrame Sample (First 5 Rows) ===")
display(complaints_df.head())

print("\n=== Unique ID Counts ===")
print(f"Unique Complaint IDs: {complaints_df['complaint_id'].nunique()}")
print(f"Unique Customer IDs: {complaints_df['customer_id'].nunique()}\n")

print("=== Duplicate Records Count ===")
print(f"Duplicate Complaint IDs count: {complaints_df['complaint_id'].duplicated().sum()}")
print(f"Total fully duplicate rows: {complaints_df.duplicated().sum()}\n")

print("=== Missing Values Count per Column ===")
display(complaints_df.isnull().sum().to_frame(name='missing_count'))

print("\n=== Leading Raw Categories ===")
if 'issue_category' in complaints_df.columns:
    display(complaints_df['issue_category'].value_counts(dropna=False).head(10))
elif 'complaint_category' in complaints_df.columns:
    display(complaints_df['complaint_category'].value_counts(dropna=False).head(10))
else:
    # If category column has a different name, display value counts for object/categorical columns
    for col in complaints_df.select_dtypes(include=['object', 'category']).columns:
        if col not in ['complaint_id', 'customer_id', 'cell_site_id']:
            print(f"Top values for {col}:")
            display(complaints_df[col].value_counts(dropna=False).head(10))


=== Complaints DataFrame Shape ===
Shape: (5000, 15) (Rows: 5000, Columns: 15)

=== DataFrame Sample (First 5 Rows) ===


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,complaint_status,sentiment_tag,network_type,device_type,amount_impacted
0,CMP000001,CUST00246,SITE0878,2026-05-04,17:04,Jaipur,Rajasthan,MyJio App,NaN,SIM activation or verification is pending afte...,Open,Urgent,VoLTE,JioPhone,799.0
1,CMP000002,CUST00793,SITE0461,2026-02-05,19:16,Ahmedabad,Gujarat,Retail Store,NaN,SIM activation or verification is pending afte...,NaN,Urgent,VoLTE,iOS,399.0
2,CMP000003,CUST00931,SITE0550,2026-02-09,20:46,Hyderabad,Andhra Pradesh & Telangana,Call Center,NaN,Fiber connection outage reported for apartment...,In Progress,Angry,5G,JioPhone,199.0
3,CMP000004,CUST00043,SITE0443,2026-02-18,13:45,Ahmedabad,Gujarat,Email,internet not working,Customer says calls drop during office hours.,Open,Frustrated,VoLTE,iOS,399.0
4,CMP000005,CUST00723,SITE0214,2026-02-01,23:28,Chennai,Tamil Nadu,Social Media,payment deducted recharge not done,Fiber connection outage reported for apartment...,closeD,Urgent,Fiber,JioPhone,399.0



=== Unique ID Counts ===
Unique Complaint IDs: 4901
Unique Customer IDs: 1218

=== Duplicate Records Count ===
Duplicate Complaint IDs count: 99
Total fully duplicate rows: 0

=== Missing Values Count per Column ===


,missing_count
complaint_id,0
customer_id,0
cell_site_id,0
complaint_date,6
complaint_time,0
city,0
circle,0
complaint_channel,0
issue_category,290
issue_description,0



=== Leading Raw Categories ===


,count
issue_category,
network down,416
recharge failed,356
SLOW DATA,326
No Signal,321
call drop,312
internet not working,299
NaN,290
network outage,267
service request pending,265


## Milestone 02 · Onboard the XLSX network source and JSON reference collections

**From the team:** Network engineering provides Excel incident logs, while customer, site, recharge, and service-request histories are organized as distinct collections inside a single JSON reference file.

**Your responsibility:** Read the workbook and JSON; materialize five independent source DataFrames and policy notes. Do not merge repeated recharge or request records prematurely.

**Required evidence:** `customers_df`, `cell_sites_df`, `network_df`, `recharge_df`, `requests_df`, `operations_notes`, and source row counts.

In [4]:
# Milestone 02 — Onboard network and customer sources
# Task 1: Parse careops_reference.json and inspect its named collections and operational notes.
# Task 2: Build customers_df and cell_sites_df from their distinct JSON collections.
# Task 3: Load the XLSX source into network_df; inspect network-report identifiers and severity.
# Task 4: Convert JSON recharge_failures and service_requests collections into recharge_df and requests_df.
# Task 5: Preserve operations_notes as a string, and display source record counts and sample rows.
# Required output: customers_df, cell_sites_df, network_df, recharge_df, requests_df, operations_notes.




In [5]:
# Milestone 02 — Onboard network and customer sources

# Task 1: Parse careops_reference.json and inspect its named collections and operational notes
with open(DATA_DIR / 'careops_reference.json', 'r', encoding='utf-8') as f:
    ref_data = json.load(f)

print("=== Top-level keys in careops_reference.json ===")
if isinstance(ref_data, dict):
    print(list(ref_data.keys()))
else:
    print(f"JSON root type: {type(ref_data)}")

def extract_collection(data, candidate_keys):
    """Safely extract a list/collection from dict using candidate key names (case-insensitive)."""
    if not isinstance(data, dict):
        return []
    lower_map = {k.lower(): v for k, v in data.items()}
    for key in candidate_keys:
        if key.lower() in lower_map and isinstance(lower_map[key.lower()], list):
            return lower_map[key.lower()]
    # Fallback: search nested dicts one level deep
    for v in data.values():
        if isinstance(v, dict):
            nested_lower = {nk.lower(): nv for nk, nv in v.items()}
            for key in candidate_keys:
                if key.lower() in nested_lower and isinstance(nested_lower[key.lower()], list):
                    return nested_lower[key.lower()]
    return []

# Task 2: Build customers_df and cell_sites_df from their distinct JSON collections
customers_raw = extract_collection(ref_data, ['customers', 'customer_profiles', 'customer_history', 'customer_records'])
cell_sites_raw = extract_collection(ref_data, ['cell_sites', 'sites', 'site_inventory', 'tower_sites', 'cell_site_inventory'])

customers_df = pd.DataFrame(customers_raw)
cell_sites_df = pd.DataFrame(cell_sites_raw)

# Task 3: Load the XLSX source into network_df; inspect network-report identifiers and severity
excel_path = DATA_DIR / 'network_issue_reports.xlsx'
xl = pd.ExcelFile(excel_path)
print("\n=== Sheet names in network_issue_reports.xlsx ===", xl.sheet_names)
network_df = pd.read_excel(excel_path, sheet_name=0)

print("\n=== Network identifiers & severity preview ===")
id_sev_cols = [c for c in network_df.columns if any(k in c.lower() for k in ['report', 'site', 'severity', 'status', 'date'])]
display(network_df[id_sev_cols].head(5) if id_sev_cols else network_df.head(5))

# Task 4: Convert JSON recharge_failures and service_requests collections into recharge_df and requests_df
recharge_raw = extract_collection(ref_data, ['recharge_failures', 'recharges', 'recharge_history', 'recharge_logs'])
requests_raw = extract_collection(ref_data, ['service_requests', 'requests', 'service_request_history'])

recharge_df = pd.DataFrame(recharge_raw)
requests_df = pd.DataFrame(requests_raw)

# Task 5: Preserve operations_notes as a string, and display source record counts and sample rows
notes_val = ""
if isinstance(ref_data, dict):
    for k in ['operations_notes', 'operational_notes', 'policy_notes', 'notes']:
        if k in ref_data and ref_data[k] is not None:
            if isinstance(ref_data[k], str):
                notes_val = ref_data[k]
            elif isinstance(ref_data[k], list):
                notes_val = "\n".join(str(item) for item in ref_data[k])
            elif isinstance(ref_data[k], dict):
                notes_val = json.dumps(ref_data[k], indent=2, ensure_ascii=False)
            else:
                notes_val = str(ref_data[k])
            break

operations_notes = str(notes_val)

print("\n=== Operations Notes ===")
print(operations_notes if operations_notes else "(No operations_notes string found in JSON)")

print("\n=== Source Record Counts ===")
counts_summary = pd.DataFrame({
    'source_name': ['customers_df', 'cell_sites_df', 'network_df', 'recharge_df', 'requests_df'],
    'rows': [len(customers_df), len(cell_sites_df), len(network_df), len(recharge_df), len(requests_df)],
    'columns': [customers_df.shape[1], cell_sites_df.shape[1], network_df.shape[1], recharge_df.shape[1], requests_df.shape[1]]
})
display(counts_summary)

for name, frame in [
    ('customers_df', customers_df),
    ('cell_sites_df', cell_sites_df),
    ('network_df', network_df),
    ('recharge_df', recharge_df),
    ('requests_df', requests_df),
]:
    print(f"\n--- {name} sample (top 3 rows) ---")
    display(frame.head(3))




=== Top-level keys in careops_reference.json ===
['dataset_description', 'customers', 'cell_sites', 'recharge_failures', 'service_requests', 'network_operations_notes']

=== Sheet names in network_issue_reports.xlsx === ['network_issue_reports']

=== Network identifiers & severity preview ===


,network_report_id,cell_site_id,report_date,severity,current_status
0,NW000001,SITE0964,2026-05-25,critical,Monitoring
1,NW000002,SITE0972,2026-05-15,Critical,Resolved
2,NW000003,SITE0646,2026-02-01,Low,Monitoring
3,NW000004,SITE0275,2026-03-08,critical,pending
4,NW000005,SITE0836,2026-05-06,critical,Open



=== Operations Notes ===
(No operations_notes string found in JSON)

=== Source Record Counts ===


,source_name,rows,columns
0,customers_df,1200,9
1,cell_sites_df,1000,9
2,network_df,1300,10
3,recharge_df,1500,9
4,requests_df,1500,10



--- customers_df sample (top 3 rows) ---


,customer_id,customer_name,customer_segment,plan_type,home_city,circle,account_age_days,preferred_service,monthly_recharge_value
0,CUST00001,Krishna Mehta,Prepaid,4G Value,Indore,Madhya Pradesh,581,Mobile Data,1499
1,CUST00002,Vivaan Sharma,Prepaid,5G Unlimited,Delhi,Delhi NCR,905,Voice,1499
2,CUST00003,Reyansh Kulkarni,Prepaid,4G Value,Lucknow,UP East,1849,Roaming,399


... (4 outputs left out)


--- recharge_df sample (top 3 rows) ---


,recharge_id,complaint_id,customer_id,recharge_date,recharge_amount,payment_mode,recharge_status,refund_status,refund_completed_date
0,RCH000001,CMP002599,CUST00947,2026-02-07,None,upi,failed payment,pending refund,None
1,RCH000002,CMP001502,CUST00264,2026-05-29,None,upi,Recharge Pending,Not Initiated,None
2,RCH000003,CMP004726,CUST00498,2026-03-30,None,Cash,FAILED,Failed,None



--- requests_df sample (top 3 rows) ---


,service_request_id,complaint_id,customer_id,request_created_date,request_closed_date,request_type,service_status,assigned_team,resolution_time_hours,escalation_flag
0,SR000001,CMP004694,CUST00217,2026-04-06 02:00,,Recharge Support,Escalated,Recharge Ops,,Y
1,SR000002,CMP001907,CUST00813,2026-03-15 00:00,,Recharge Support,Open,Fiber Field Team,56,yes
2,SR000003,CMP003243,CUST00377,2026-05-07 21:00,2026-05-08 08:00,Billing Investigation,Closed,Care Escalation Desk,,N


## Milestone 03 · Validate and standardize complaint records

**From the team:** Quality assurance warns that missing issue labels, mixed-case statuses, and invalid timestamps make complaint-age comparisons unreliable.

**Your responsibility:** Deduplicate only by complaint_id; standardize categories and statuses; parse dates and monetary fields. Record missing customer/site references and unparseable dates without erasing valid multi-complaint histories.

**Handover evidence:** Create `complaints_clean`, `quality_summary` and normalized `issue_group` with complaint-level uniqueness.

In [6]:
# Milestone 03 — Validate and standardize complaint records
# Task 1: Deduplicate only by complaint_id; standardize categories and statuses; parse dates and monetary fields. Record missing customer/site references and unparseable dates without erasing valid multi-complaint histories.
# Task 2: Produce this required evidence: Create `complaints_clean`, `quality_summary` and normalized `issue_group` with complaint-level uniqueness.
# Implement your code below and display the requested outputs.



In [7]:
# Milestone 03 — Validate and standardize complaint records

df_c = complaints_df.copy()

# 1. Deduplicate only by complaint_id (preserving valid multi-complaint customer histories)
raw_rows = len(df_c)
dup_complaint_ids_before = int(df_c['complaint_id'].duplicated().sum())
df_c = df_c.drop_duplicates(subset=['complaint_id'], keep='first').copy()
clean_rows = len(df_c)

# 2. Clean text/categorical fields safely
def clean_str_series(s):
    return s.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA})

for col in ['complaint_id', 'customer_id', 'cell_site_id', 'city', 'circle',
            'complaint_channel', 'issue_category', 'issue_description',
            'complaint_status', 'sentiment_tag', 'network_type', 'device_type']:
    if col in df_c.columns:
        df_c[col] = clean_str_series(df_c[col])

# Standardize complaint_status (e.g., 'closeD' -> 'Closed', 'open' -> 'Open')
status_map = {
    'open': 'Open',
    'in progress': 'In Progress',
    'in_progress': 'In Progress',
    'pending': 'Pending',
    'closed': 'Closed',
    'resolved': 'Resolved',
    'escalated': 'Escalated'
}
df_c['complaint_status'] = (
    df_c['complaint_status']
    .str.lower()
    .map(status_map)
    .fillna(df_c['complaint_status'].str.title())
)

# Normalize issue_category into a clean `issue_group`
norm_cat = (
    df_c['issue_category']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

category_group_map = {
    'network down': 'Network & Connectivity',
    'network outage': 'Network & Connectivity',
    'no signal': 'Network & Connectivity',
    'slow data': 'Network & Connectivity',
    'internet not working': 'Network & Connectivity',
    'call drop': 'Network & Connectivity',
    'recharge failed': 'Recharge & Billing',
    'payment deducted recharge not done': 'Recharge & Billing',
    'service request pending': 'Service Request & Activation',
    'sim activation pending': 'Service Request & Activation',
}

def map_issue_group(val):
    if pd.isna(val):
        return 'Unknown / Uncategorized'
    if val in category_group_map:
        return category_group_map[val]
    if any(k in val for k in ['network', 'signal', 'data', 'internet', 'call', 'fiber', 'tower', 'outage', 'coverage']):
        return 'Network & Connectivity'
    if any(k in val for k in ['recharge', 'payment', 'refund', 'billing', 'amount', 'deduct']):
        return 'Recharge & Billing'
    if any(k in val for k in ['sim', 'service', 'request', 'activation', 'port', 'kyc']):
        return 'Service Request & Activation'
    return val.title()

df_c['issue_group'] = norm_cat.map(map_issue_group)

# 3. Parse complaint_date (and complaint_time if present) into real datetime without dropping unparseable rows
parsed_date = pd.to_datetime(df_c['complaint_date'], errors='coerce')
if 'complaint_time' in df_c.columns:
    combined_dt_str = (
        df_c['complaint_date'].astype('string').fillna('') + ' ' +
        df_c['complaint_time'].astype('string').fillna('')
    ).str.strip()
    parsed_dt = pd.to_datetime(combined_dt_str, errors='coerce')
    df_c['complaint_datetime'] = parsed_dt.fillna(parsed_date)
else:
    df_c['complaint_datetime'] = parsed_date

df_c['complaint_date'] = parsed_date
df_c['unparseable_date_flag'] = df_c['complaint_date'].isna()

# 4. Parse monetary field (amount_impacted) cleanly
if 'amount_impacted' in df_c.columns:
    cleaned_amt = (
        df_c['amount_impacted']
        .astype('string')
        .str.replace(r'[₹,\s]', '', regex=True)
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )
    df_c['amount_impacted'] = pd.to_numeric(cleaned_amt, errors='coerce')

# 5. Record missing customer/site references (if customers_df / cell_sites_df exist in scope)
valid_cust_ids = set(customers_df['customer_id'].dropna()) if 'customers_df' in globals() and 'customer_id' in customers_df.columns else set()
valid_site_ids = set(cell_sites_df['cell_site_id'].dropna()) if 'cell_sites_df' in globals() and 'cell_sites_id' in cell_sites_df.columns else set()

df_c['missing_customer_id_flag'] = df_c['customer_id'].isna()
df_c['missing_site_id_flag'] = df_c['cell_site_id'].isna()
df_c['unmatched_customer_ref_flag'] = df_c['customer_id'].notna() & (~df_c['customer_id'].isin(valid_cust_ids)) if valid_cust_ids else False
df_c['unmatched_site_ref_flag'] = df_c['cell_site_id'].notna() & (~df_c['cell_site_id'].isin(valid_site_ids)) if valid_site_ids else False

# Finalize complaints_clean with unique complaint_id
complaints_clean = df_c.copy()

# 6. Build quality_summary DataFrame
quality_summary = pd.DataFrame({
    'metric': [
        'raw_complaint_rows',
        'duplicate_complaint_ids_removed',
        'clean_unique_complaints',
        'unique_customers_in_clean',
        'missing_complaint_date_or_unparseable',
        'missing_issue_category',
        'missing_complaint_status',
        'missing_amount_impacted',
        'missing_customer_id',
        'missing_site_id',
        'unmatched_customer_ref',
        'unmatched_site_ref'
    ],
    'count': [
        raw_rows,
        dup_complaint_ids_before,
        clean_rows,
        int(complaints_clean['customer_id'].nunique(dropna=True)),
        int(complaints_clean['unparseable_date_flag'].sum()),
        int(complaints_clean['issue_category'].isna().sum()),
        int(complaints_clean['complaint_status'].isna().sum()),
        int(complaints_clean['amount_impacted'].isna().sum()) if 'amount_impacted' in complaints_clean.columns else 0,
        int(complaints_clean['missing_customer_id_flag'].sum()),
        int(complaints_clean['missing_site_id_flag'].sum()),
        int(complaints_clean['unmatched_customer_ref_flag'].sum()),
        int(complaints_clean['unmatched_site_ref_flag'].sum())
    ]
})

print("=== Milestone 03 Quality Summary ===")
display(quality_summary)

print("\n=== Normalized issue_group Counts ===")
display(complaints_clean['issue_group'].value_counts(dropna=False).to_frame(name='complaint_count'))

print("\n=== Standardized complaint_status Counts ===")
display(complaints_clean['complaint_status'].value_counts(dropna=False).to_frame(name='complaint_count'))

print("\n=== complaints_clean Preview ===")
display(complaints_clean.head(5))

=== Milestone 03 Quality Summary ===


,metric,count
0,raw_complaint_rows,5000
1,duplicate_complaint_ids_removed,99
2,clean_unique_complaints,4901
3,unique_customers_in_clean,1217
4,missing_complaint_date_or_unparseable,12
5,missing_issue_category,282
6,missing_complaint_status,452
7,missing_amount_impacted,425
8,missing_customer_id,0
9,missing_site_id,0



=== Normalized issue_group Counts ===


,complaint_count
issue_group,
Network & Connectivity,2841
Recharge & Billing,1187
Service Request & Activation,453
Unknown / Uncategorized,282
Complaint Unresolved,138



=== Standardized complaint_status Counts ===


,complaint_count
complaint_status,
Closed,922
Open,864
Pending Review,474
Escalated,453
<NA>,452
Resolved,448
Pending,444
In-Progress,443
In Progress,401



=== complaints_clean Preview ===


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,...,network_type,device_type,amount_impacted,issue_group,complaint_datetime,unparseable_date_flag,missing_customer_id_flag,missing_site_id_flag,unmatched_customer_ref_flag,unmatched_site_ref_flag
0,CMP000001,CUST00246,SITE0878,2026-05-04,17:04,Jaipur,Rajasthan,MyJio App,<NA>,SIM activation or verification is pending afte...,...,VoLTE,JioPhone,799.0,Unknown / Uncategorized,2026-05-04 17:04:00,False,False,False,False,False
1,CMP000002,CUST00793,SITE0461,2026-02-05,19:16,Ahmedabad,Gujarat,Retail Store,<NA>,SIM activation or verification is pending afte...,...,VoLTE,iOS,399.0,Unknown / Uncategorized,2026-02-05 19:16:00,False,False,False,False,False
2,CMP000003,CUST00931,SITE0550,2026-02-09,20:46,Hyderabad,Andhra Pradesh & Telangana,Call Center,<NA>,Fiber connection outage reported for apartment...,...,5G,JioPhone,199.0,Unknown / Uncategorized,2026-02-09 20:46:00,False,False,False,False,False
3,CMP000004,CUST00043,SITE0443,2026-02-18,13:45,Ahmedabad,Gujarat,Email,internet not working,Customer says calls drop during office hours.,...,VoLTE,iOS,399.0,Network & Connectivity,2026-02-18 13:45:00,False,False,False,False,False
4,CMP000005,CUST00723,SITE0214,2026-02-01,23:28,Chennai,Tamil Nadu,Social Media,payment deducted recharge not done,Fiber connection outage reported for apartment...,...,Fiber,JioPhone,399.0,Recharge & Billing,2026-02-01 23:28:00,False,False,False,False,False


## Milestone 04 · Prepare reliable network incident signals

**From the team:** Network operations wants tower status and incident severity contextualized before escalation decisions.

**Your responsibility:** Deduplicate network report IDs; normalize severity/status; aggregate distinct incidents at cell-site level; join site inventory without multiplying incidents.

**Handover evidence:** Create `network_clean`, `network_by_site`, and `sites_clean` with unique site IDs.

In [8]:
# Milestone 04 — Prepare reliable network incident signals
# Task 1: Deduplicate network report IDs; normalize severity/status; aggregate distinct incidents at cell-site level; join site inventory without multiplying incidents.
# Task 2: Produce this required evidence: Create `network_clean`, `network_by_site`, and `sites_clean` with unique site IDs.
# Implement your code below and display the requested outputs.



In [9]:
# Milestone 04 — Prepare reliable network incident signals

# 1. Clean cell_sites_df -> sites_clean (unique cell_site_id)
df_s = cell_sites_df.copy()

def clean_str_col(s):
    return s.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA})

for col in df_s.select_dtypes(include=['object', 'string']).columns:
    df_s[col] = clean_str_col(df_s[col])

if 'cell_site_id' in df_s.columns:
    df_s = df_s.dropna(subset=['cell_site_id']).drop_duplicates(subset=['cell_site_id'], keep='first').copy()

if 'site_status' in df_s.columns:
    df_s['site_status'] = df_s['site_status'].str.title()

for col in ['avg_signal_strength_dbm', 'capacity_utilization_pct']:
    if col in df_s.columns:
        df_s[col] = pd.to_numeric(df_s[col], errors='coerce')

if 'last_maintenance_date' in df_s.columns:
    df_s['last_maintenance_date'] = pd.to_datetime(df_s['last_maintenance_date'], errors='coerce')

sites_clean = df_s.copy()

# 2. Clean network_df -> network_clean (deduplicate by network_report_id, parse dates, normalize severity & status)
df_n = network_df.copy()

for col in df_n.select_dtypes(include=['object', 'string']).columns:
    df_n[col] = clean_str_col(df_n[col])

raw_net_rows = len(df_n)
dup_net_ids = int(df_n['network_report_id'].duplicated().sum()) if 'network_report_id' in df_n.columns else 0
if 'network_report_id' in df_n.columns:
    df_n = df_n.drop_duplicates(subset=['network_report_id'], keep='first').copy()

if 'report_date' in df_n.columns:
    df_n['report_date'] = pd.to_datetime(df_n['report_date'], errors='coerce')

for col in ['affected_customers', 'avg_downtime_minutes']:
    if col in df_n.columns:
        df_n[col] = pd.to_numeric(df_n[col], errors='coerce')

# Normalize severity & status
severity_map = {
    'critical': 'Critical',
    'high': 'High',
    'major': 'High',
    'medium': 'Medium',
    'moderate': 'Medium',
    'low': 'Low',
    'minor': 'Low'
}
if 'severity' in df_n.columns:
    df_n['severity_norm'] = (
        df_n['severity']
        .str.lower()
        .str.strip()
        .map(severity_map)
        .fillna(df_n['severity'].str.title())
    )
    df_n['severity'] = df_n['severity_norm']
    df_n['is_severe_flag'] = df_n['severity'].isin(['Critical', 'High'])
else:
    df_n['is_severe_flag'] = False

status_col = 'current_status' if 'current_status' in df_n.columns else ('status' if 'status' in df_n.columns else None)
if status_col:
    df_n[status_col] = df_n[status_col].str.strip().str.title()
    df_n['is_open_incident_flag'] = ~df_n[status_col].isin(['Resolved', 'Closed'])
    df_n['is_open_severe_flag'] = df_n['is_severe_flag'] & df_n['is_open_incident_flag']
else:
    df_n['is_open_incident_flag'] = True
    df_n['is_open_severe_flag'] = df_n['is_severe_flag']

network_clean = df_n.copy()

# 3. Aggregate distinct incidents at cell-site level -> network_by_site
net_valid = network_clean.dropna(subset=['cell_site_id']).copy()

agg_dict = {
    'network_report_id': 'count',
    'is_severe_flag': 'sum',
    'is_open_incident_flag': 'sum',
    'is_open_severe_flag': 'sum',
}
if 'affected_customers' in net_valid.columns:
    agg_dict['affected_customers'] = 'sum'
if 'avg_downtime_minutes' in net_valid.columns:
    agg_dict['avg_downtime_minutes'] = 'mean'
if 'report_date' in net_valid.columns:
    agg_dict['report_date'] = 'max'

grouped = net_valid.groupby('cell_site_id', as_index=False).agg(agg_dict)

rename_map = {
    'network_report_id': 'network_report_count',
    'is_severe_flag': 'severe_network_report_count',
    'is_open_incident_flag': 'open_network_report_count',
    'is_open_severe_flag': 'open_severe_network_report_count',
    'affected_customers': 'total_affected_customers',
    'avg_downtime_minutes': 'mean_downtime_minutes',
    'report_date': 'latest_report_date',
}
grouped = grouped.rename(columns=rename_map)

for c in ['network_report_count', 'severe_network_report_count', 'open_network_report_count', 'open_severe_network_report_count']:
    grouped[c] = grouped[c].astype(int)

# Join site inventory without multiplying incidents
network_by_site = grouped.merge(sites_clean, on='cell_site_id', how='left')

print("=== Milestone 04 Summary ===")
print(f"sites_clean shape: {sites_clean.shape} | Unique cell_site_id: {sites_clean['cell_site_id'].nunique()}")
print(f"network_clean shape: {network_clean.shape} (Raw rows: {raw_net_rows}, Dup IDs removed: {dup_net_ids})")
print(f"network_by_site shape: {network_by_site.shape} | Total reports summed: {network_by_site['network_report_count'].sum()}")

print("\n=== Severity & Status Breakdown in network_clean ===")
display(network_clean['severity'].value_counts(dropna=False).to_frame(name='report_count'))
if status_col:
    display(network_clean[status_col].value_counts(dropna=False).to_frame(name='report_count'))

print("\n=== network_by_site Preview (Top 5) ===")
display(network_by_site.head(5))

=== Milestone 04 Summary ===
sites_clean shape: (1000, 9) | Unique cell_site_id: 1000
network_clean shape: (1300, 14) (Raw rows: 1300, Dup IDs removed: 0)
network_by_site shape: (721, 16) | Total reports summed: 1300

=== Severity & Status Breakdown in network_clean ===


,report_count
severity,
High,381
Critical,371
Low,201
Medium,174
Major Outage,173


,report_count
current_status,
Open,247
Monitoring,234
Escalated,222
Resolved,210
Pending,201
Closed,186



=== network_by_site Preview (Top 5) ===


,cell_site_id,network_report_count,severe_network_report_count,open_network_report_count,open_severe_network_report_count,total_affected_customers,mean_downtime_minutes,latest_report_date,site_name,city,circle,tower_type,site_status,avg_signal_strength_dbm,capacity_utilization_pct,last_maintenance_date
0,SITE0005,1,1,1,1,1027,516.0,2026-02-01,MUM-NW-0005,Mumbai,Mumbai,Indoor Booster,Active,-115.5,57.9,2025-10-29
1,SITE0008,2,1,1,1,1217,445.0,2026-04-26,AHM-NW-0008,Ahmedabad,Gujarat,Indoor Booster,Active,-90.7,56.9,2026-02-13
2,SITE0010,2,1,0,0,1670,375.0,2026-05-10,AHM-NW-0010,Ahmedabad,Gujarat,Fiber Hub,Active,-89.4,53.1,2025-12-10
3,SITE0011,2,2,2,2,1432,429.5,2026-04-08,LUC-NW-0011,Lucknow,UP East,Fiber Hub,Under Maintenance,-95.6,40.4,2026-04-11
4,SITE0013,3,1,2,0,4447,394.0,2026-05-14,PUN-NW-0013,Pune,Maharashtra,Small Cell,Congested,-97.7,49.0,2025-07-15


## Milestone 05 · Audit recharge failures and refund risk

**From the team:** Recharge Support needs to spot unresolved money-impacting cases and failed payments linked to complaints.

**Your responsibility:** Normalize recharge status, parse amounts and dates, flag failed/pending outcomes and pending refunds, and aggregate by complaint ID. Do not discard valid multiple recharge events.

**Handover evidence:** Create `recharge_clean`, `recharge_by_complaint` and readable risk counts.

In [10]:
# Milestone 05 — Audit recharge failures and refund risk
# Task 1: Normalize recharge status, parse amounts and dates, flag failed/pending outcomes and pending refunds, and aggregate by complaint ID. Do not discard valid multiple recharge events.
# Task 2: Produce this required evidence: Create `recharge_clean`, `recharge_by_complaint` and readable risk counts.
# Implement your code below and display the requested outputs.



In [11]:
# Milestone 05 — Audit recharge failures and refund risk

df_r = recharge_df.copy()

def clean_str_r(s):
    return s.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA, 'null': pd.NA})

for col in df_r.select_dtypes(include=['object', 'string']).columns:
    df_r[col] = clean_str_r(df_r[col])

# 1. Normalize recharge_status and refund_status
status_col_r = 'recharge_status' if 'recharge_status' in df_r.columns else ('status' if 'status' in df_r.columns else None)
refund_col_r = 'refund_status' if 'refund_status' in df_r.columns else None

recharge_status_map = {
    'failed': 'Failed',
    'failed payment': 'Failed',
    'payment failed': 'Failed',
    'recharge failed': 'Failed',
    'pending': 'Pending',
    'recharge pending': 'Pending',
    'in progress': 'Pending',
    'success': 'Success',
    'successful': 'Success',
    'completed': 'Success'
}

if status_col_r:
    df_r['recharge_status_clean'] = (
        df_r[status_col_r]
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .map(recharge_status_map)
        .fillna(df_r[status_col_r].str.title())
    )
    df_r['recharge_status'] = df_r['recharge_status_clean']
    df_r['is_failed_recharge_flag'] = df_r['recharge_status'].str.lower().str.contains('fail', na=False)
    df_r['is_pending_recharge_flag'] = df_r['recharge_status'].str.lower().str.contains('pend', na=False)
    df_r['is_failed_or_pending_flag'] = df_r['is_failed_recharge_flag'] | df_r['is_pending_recharge_flag']
else:
    df_r['is_failed_recharge_flag'] = False
    df_r['is_pending_recharge_flag'] = False
    df_r['is_failed_or_pending_flag'] = False

if refund_col_r:
    df_r['refund_status'] = df_r[refund_col_r].str.strip().str.title()
    refund_lower = df_r['refund_status'].str.lower()
    df_r['is_pending_refund_flag'] = refund_lower.str.contains('pend|not initiated|in progress|open|fail', na=False)
    df_r['is_refund_completed_flag'] = refund_lower.str.contains('complet|processed|refunded|success|done', na=False)
else:
    df_r['is_pending_refund_flag'] = False
    df_r['is_refund_completed_flag'] = False

df_r['at_risk_recharge_flag'] = df_r['is_failed_or_pending_flag'] | df_r['is_pending_refund_flag']

# 2. Parse amounts and dates cleanly
amt_col_r = next((c for c in ['recharge_amount', 'amount', 'txn_amount'] if c in df_r.columns), None)
if amt_col_r:
    cleaned_r_amt = (
        df_r[amt_col_r]
        .astype('string')
        .str.replace(r'[₹,\s]', '', regex=True)
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )
    df_r['recharge_amount'] = pd.to_numeric(cleaned_r_amt, errors='coerce')
else:
    df_r['recharge_amount'] = pd.NA

for dcol in ['recharge_date', 'refund_completed_date', 'created_date']:
    if dcol in df_r.columns:
        df_r[dcol] = pd.to_datetime(df_r[dcol], errors='coerce')

df_r['at_risk_amount'] = df_r['recharge_amount'].where(df_r['at_risk_recharge_flag'], 0)
df_r['pending_refund_amount'] = df_r['recharge_amount'].where(df_r['is_pending_refund_flag'], 0)

recharge_clean = df_r.copy()

# 3. Aggregate by complaint_id without discarding multiple recharge events
r_valid = recharge_clean.dropna(subset=['complaint_id']).copy()

r_agg_dict = {
    'recharge_id': 'count' if 'recharge_id' in r_valid.columns else 'size',
    'is_failed_recharge_flag': 'sum',
    'is_pending_recharge_flag': 'sum',
    'is_failed_or_pending_flag': 'sum',
    'is_pending_refund_flag': 'sum',
    'at_risk_recharge_flag': 'sum',
    'recharge_amount': 'sum',
    'at_risk_amount': 'sum',
    'pending_refund_amount': 'sum',
}
if 'recharge_date' in r_valid.columns:
    r_agg_dict['recharge_date'] = 'max'

r_grouped = r_valid.groupby('complaint_id', as_index=False).agg(r_agg_dict)

r_rename = {
    'recharge_id': 'recharge_event_count',
    'is_failed_recharge_flag': 'failed_recharge_count',
    'is_pending_recharge_flag': 'pending_recharge_count',
    'is_failed_or_pending_flag': 'failed_or_pending_recharge_count',
    'is_pending_refund_flag': 'pending_refund_count',
    'at_risk_recharge_flag': 'at_risk_recharge_count',
    'recharge_amount': 'total_recharge_amount',
    'at_risk_amount': 'total_at_risk_amount',
    'pending_refund_amount': 'total_pending_refund_amount',
    'recharge_date': 'latest_recharge_date',
}
recharge_by_complaint = r_grouped.rename(columns=r_rename)

for c in [
    'recharge_event_count', 'failed_recharge_count', 'pending_recharge_count',
    'failed_or_pending_recharge_count', 'pending_refund_count', 'at_risk_recharge_count'
]:
    recharge_by_complaint[c] = recharge_by_complaint[c].astype(int)

recharge_by_complaint['has_failed_or_pending_recharge'] = recharge_by_complaint['failed_or_pending_recharge_count'] > 0
recharge_by_complaint['has_pending_refund'] = recharge_by_complaint['pending_refund_count'] > 0
recharge_by_complaint['has_recharge_risk'] = recharge_by_complaint['at_risk_recharge_count'] > 0

# Readable risk counts summary
risk_summary = pd.DataFrame({
    'risk_metric': [
        'total_clean_recharge_records',
        'failed_recharge_records',
        'pending_recharge_records',
        'failed_or_pending_recharge_records',
        'pending_refund_records',
        'total_at_risk_recharge_records',
        'complaints_with_recharge_events',
        'complaints_with_failed_or_pending_recharge',
        'complaints_with_pending_refund',
        'total_at_risk_amount_inr',
        'total_pending_refund_amount_inr'
    ],
    'value': [
        len(recharge_clean),
        int(recharge_clean['is_failed_recharge_flag'].sum()),
        int(recharge_clean['is_pending_recharge_flag'].sum()),
        int(recharge_clean['is_failed_or_pending_flag'].sum()),
        int(recharge_clean['is_pending_refund_flag'].sum()),
        int(recharge_clean['at_risk_recharge_flag'].sum()),
        len(recharge_by_complaint),
        int(recharge_by_complaint['has_failed_or_pending_recharge'].sum()),
        int(recharge_by_complaint['has_pending_refund'].sum()),
        float(pd.to_numeric(recharge_clean['at_risk_amount'], errors='coerce').fillna(0).sum()),
        float(pd.to_numeric(recharge_clean['pending_refund_amount'], errors='coerce').fillna(0).sum())
    ]
})

print("=== Milestone 05 Recharge & Refund Risk Summary ===")
display(risk_summary)

print("\n=== Normalized Recharge Status Counts ===")
display(recharge_clean['recharge_status'].value_counts(dropna=False).to_frame(name='count'))

if refund_col_r:
    print("\n=== Refund Status Counts ===")
    display(recharge_clean['refund_status'].value_counts(dropna=False).to_frame(name='count'))

print("\n=== recharge_by_complaint Preview (Top 5) ===")
display(recharge_by_complaint.head(5))

=== Milestone 05 Recharge & Refund Risk Summary ===


,risk_metric,value
0,total_clean_recharge_records,1500.0
1,failed_recharge_records,641.0
2,pending_recharge_records,217.0
3,failed_or_pending_recharge_records,858.0
4,pending_refund_records,1037.0
5,total_at_risk_recharge_records,1303.0
6,complaints_with_recharge_events,1257.0
7,complaints_with_failed_or_pending_recharge,762.0
8,complaints_with_pending_refund,926.0
9,total_at_risk_amount_inr,388385.0



=== Normalized Recharge Status Counts ===


,count
recharge_status,
Failed,641
Payment Deducted,228
Pending,217
Success Reversed,209
<NA>,205



=== Refund Status Counts ===


,count
refund_status,
Not Initiated,279
Pending,263
Pending Refund,251
Failed,244
Completed,244
<NA>,219



=== recharge_by_complaint Preview (Top 5) ===


,complaint_id,recharge_event_count,failed_recharge_count,pending_recharge_count,failed_or_pending_recharge_count,pending_refund_count,at_risk_recharge_count,total_recharge_amount,total_at_risk_amount,total_pending_refund_amount,latest_recharge_date,has_failed_or_pending_recharge,has_pending_refund,has_recharge_risk
0,CMP000001,1,1,0,1,0,1,0,0,0,2026-04-28,True,False,True
1,CMP000003,1,0,0,0,1,1,999,999,999,2026-03-10,False,True,True
2,CMP000007,1,0,1,1,1,1,599,599,599,2026-04-11,True,True,True
3,CMP000009,1,0,1,1,1,1,149,149,149,2026-05-20,True,True,True
4,CMP000010,1,0,1,1,1,1,799,799,799,2026-03-16,True,True,True


## Milestone 06 · Detect service request and SLA follow-ups

**From the team:** The Service Desk wants unresolved or escalated requests visible on the original complaint without introducing duplicate complaint rows.

**Your responsibility:** Normalize service request statuses, parse request dates and resolution hours, distinguish still-open from closed cases, and aggregate by complaint before merging.

**Handover evidence:** Create `requests_clean`, `requests_by_complaint`; show unresolved and escalated counts and orphan references.

In [12]:
# Milestone 06 — Detect service request and SLA follow-ups
# Task 1: Normalize service request statuses, parse request dates and resolution hours, distinguish still-open from closed cases, and aggregate by complaint before merging.
# Task 2: Produce this required evidence: Create `requests_clean`, `requests_by_complaint`; show unresolved and escalated counts and orphan references.
# Implement your code below and display the requested outputs.


In [13]:
# Milestone 06 — Detect service request and SLA follow-ups

df_req = requests_df.copy()

def clean_str_req(s):
    return s.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA, 'null': pd.NA})

for col in df_req.select_dtypes(include=['object', 'string']).columns:
    df_req[col] = clean_str_req(df_req[col])

# 1. Normalize service request statuses and escalation flag
status_col_req = 'service_status' if 'service_status' in df_req.columns else ('status' if 'status' in df_req.columns else None)

req_status_map = {
    'open': 'Open',
    'in progress': 'In Progress',
    'in_progress': 'In Progress',
    'pending': 'Pending',
    'escalated': 'Escalated',
    'closed': 'Closed',
    'resolved': 'Resolved',
    'completed': 'Closed'
}

if status_col_req:
    df_req['service_status'] = (
        df_req[status_col_req]
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .map(req_status_map)
        .fillna(df_req[status_col_req].str.title())
    )
    df_req['is_open_or_unresolved_req_flag'] = ~df_req['service_status'].isin(['Closed', 'Resolved'])
else:
    df_req['is_open_or_unresolved_req_flag'] = True

if 'escalation_flag' in df_req.columns:
    esc_str = df_req['escalation_flag'].astype('string').str.strip().str.lower()
    df_req['is_escalated_req_flag'] = (
        esc_str.isin(['y', 'yes', 'true', '1', 'escalated'])
        | (df_req['service_status'].astype('string').str.lower() == 'escalated')
    )
else:
    df_req['is_escalated_req_flag'] = df_req['service_status'].astype('string').str.lower() == 'escalated'

# 2. Parse request dates and resolution_time_hours
for dcol in ['request_created_date', 'request_closed_date', 'created_date', 'closed_date']:
    if dcol in df_req.columns:
        df_req[dcol] = pd.to_datetime(df_req[dcol], errors='coerce')

if 'resolution_time_hours' in df_req.columns:
    res_clean = (
        df_req['resolution_time_hours']
        .astype('string')
        .str.replace(r'[^0-9.\-]', '', regex=True)
        .replace({'': pd.NA, '-': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )
    df_req['resolution_time_hours'] = pd.to_numeric(res_clean, errors='coerce')
else:
    df_req['resolution_time_hours'] = pd.NA

# Fallback calculation of elapsed/resolution hours from dates when available
if 'request_created_date' in df_req.columns:
    end_dt = df_req['request_closed_date'] if 'request_closed_date' in df_req.columns else pd.Series(pd.NaT, index=df_req.index)
    computed_hours = (end_dt - df_req['request_created_date']).dt.total_seconds() / 3600.0
    df_req['effective_resolution_hours'] = df_req['resolution_time_hours'].fillna(computed_hours)
else:
    df_req['effective_resolution_hours'] = df_req['resolution_time_hours']

df_req['needs_sla_followup_flag'] = df_req['is_open_or_unresolved_req_flag'] | df_req['is_escalated_req_flag']

# Detect orphan references (complaint_id missing or not found in complaints_clean)
valid_complaint_ids = set(complaints_clean['complaint_id'].dropna()) if 'complaints_clean' in globals() and 'complaint_id' in complaints_clean.columns else set()
df_req['is_orphan_complaint_ref'] = (
    df_req['complaint_id'].isna()
    | (~df_req['complaint_id'].isin(valid_complaint_ids) if valid_complaint_ids else False)
)

requests_clean = df_req.copy()

# 3. Aggregate by complaint_id before merging (preserving multiple service requests per complaint)
req_valid = requests_clean.dropna(subset=['complaint_id']).copy()

req_agg_dict = {
    'service_request_id': 'count' if 'service_request_id' in req_valid.columns else 'size',
    'is_open_or_unresolved_req_flag': 'sum',
    'is_escalated_req_flag': 'sum',
    'needs_sla_followup_flag': 'sum',
    'effective_resolution_hours': 'mean',
}
if 'request_created_date' in req_valid.columns:
    req_agg_dict['request_created_date'] = 'max'

req_grouped = req_valid.groupby('complaint_id', as_index=False).agg(req_agg_dict)

requests_by_complaint = req_grouped.rename(columns={
    'service_request_id': 'service_request_count',
    'is_open_or_unresolved_req_flag': 'unresolved_request_count',
    'is_escalated_req_flag': 'escalated_request_count',
    'needs_sla_followup_flag': 'sla_followup_request_count',
    'effective_resolution_hours': 'avg_resolution_hours',
    'request_created_date': 'latest_request_created_date',
})

for c in ['service_request_count', 'unresolved_request_count', 'escalated_request_count', 'sla_followup_request_count']:
    requests_by_complaint[c] = requests_by_complaint[c].astype(int)

requests_by_complaint['has_unresolved_request'] = requests_by_complaint['unresolved_request_count'] > 0
requests_by_complaint['has_escalated_request'] = requests_by_complaint['escalated_request_count'] > 0
requests_by_complaint['needs_sla_followup'] = requests_by_complaint['sla_followup_request_count'] > 0

# Summary counts & orphan references
req_summary = pd.DataFrame({
    'metric': [
        'total_clean_service_requests',
        'unresolved_or_open_requests',
        'escalated_requests',
        'requests_needing_sla_followup',
        'orphan_complaint_references',
        'complaints_with_service_requests',
        'complaints_with_unresolved_requests',
        'complaints_with_escalated_requests'
    ],
    'count': [
        len(requests_clean),
        int(requests_clean['is_open_or_unresolved_req_flag'].sum()),
        int(requests_clean['is_escalated_req_flag'].sum()),
        int(requests_clean['needs_sla_followup_flag'].sum()),
        int(requests_clean['is_orphan_complaint_ref'].sum()),
        len(requests_by_complaint),
        int(requests_by_complaint['has_unresolved_request'].sum()),
        int(requests_by_complaint['has_escalated_request'].sum())
    ]
})

print("=== Milestone 06 Service Request & SLA Summary ===")
display(req_summary)

print("\n=== Orphan Service Request References (Sample up to 10) ===")
orphan_reqs = requests_clean[requests_clean['is_orphan_complaint_ref']]
display(orphan_reqs.head(10))

print("\n=== requests_by_complaint Preview (Top 5) ===")
display(requests_by_complaint.head(5))

=== Milestone 06 Service Request & SLA Summary ===


,metric,count
0,total_clean_service_requests,1500
1,unresolved_or_open_requests,1089
2,escalated_requests,840
3,requests_needing_sla_followup,1284
4,orphan_complaint_references,3
5,complaints_with_service_requests,1265
6,complaints_with_unresolved_requests,958
7,complaints_with_escalated_requests,763



=== Orphan Service Request References (Sample up to 10) ===


,service_request_id,complaint_id,customer_id,request_created_date,request_closed_date,request_type,service_status,assigned_team,resolution_time_hours,escalation_flag,is_open_or_unresolved_req_flag,is_escalated_req_flag,effective_resolution_hours,needs_sla_followup_flag,is_orphan_complaint_ref
325,SR000326,CMPX943667,CUST00055,2026-05-10 10:00:00,NaT,Recharge Support,Pending,Retail Ops,108,<NA>,True,False,108,True,True
1138,SR001139,CMPX905803,CUST01012,2026-03-31 15:00:00,NaT,Billing Investigation,<NA>,Circle Operations,86,Y,True,True,86,True,True
1492,SR001493,CMPX928838,CUST00696,2026-03-20 05:00:00,NaT,Network Visit,Open,Fiber Field Team,<NA>,N,True,False,<NA>,True,True



=== requests_by_complaint Preview (Top 5) ===


,complaint_id,service_request_count,unresolved_request_count,escalated_request_count,sla_followup_request_count,avg_resolution_hours,latest_request_created_date,has_unresolved_request,has_escalated_request,needs_sla_followup
0,CMP000002,1,0,1,1,33.0,2026-04-30 18:00:00,False,True,True
1,CMP000003,1,0,0,0,102.0,2026-02-24 03:00:00,False,False,False
2,CMP000008,1,1,0,1,<NA>,2026-05-19 17:00:00,True,False,True
3,CMP000009,2,2,1,2,19.0,2026-05-05 08:00:00,True,True,True
4,CMP000011,1,1,0,1,95.0,2026-03-30 17:00:00,True,False,True


## Milestone 07 · Analyze city, circle and repeated customer impact

**From the team:** Circle managers need geographic clusters and repeat contacts—not misleading counts created by multi-table joins.

**Your responsibility:** Use cleaned complaint facts and site context to summarize cities/circles and repeated customer complaints; preserve missing site references for review.

**Handover evidence:** Create `location_summary`, `repeat_customers` and `customer_repeat_count` at complaint grain.

In [14]:
# Milestone 07 — Analyze city, circle and repeated customer impact
# Task 1: Use cleaned complaint facts and site context to summarize cities/circles and repeated customer complaints; preserve missing site references for review.
# Task 2: Produce this required evidence: Create `location_summary`, `repeat_customers` and `customer_repeat_count` at complaint grain.
# Implement your code below and display the requested outputs.


In [15]:
# Milestone 07 — Analyze city, circle and repeated customer impact

# 1. Compute customer_repeat_count at complaint grain without losing any complaint rows
cust_counts = (
    complaints_clean.dropna(subset=['customer_id'])
    .groupby('customer_id')['complaint_id']
    .nunique()
)

customer_repeat_count = (
    complaints_clean['customer_id']
    .map(cust_counts)
    .fillna(0)
    .astype(int)
)
complaints_clean['customer_repeat_count'] = customer_repeat_count

# 2. Build repeat_customers summary table (customers with > 1 complaint)
repeat_cust_df = (
    complaints_clean.dropna(subset=['customer_id'])
    .groupby('customer_id', as_index=False)
    .agg(
        total_complaints=('complaint_id', 'nunique'),
        total_amount_impacted=('amount_impacted', 'sum'),
        first_complaint_date=('complaint_datetime', 'min'),
        latest_complaint_date=('complaint_datetime', 'max'),
        cities=('city', lambda s: ', '.join(sorted(s.dropna().unique()))),
        circles=('circle', lambda s: ', '.join(sorted(s.dropna().unique()))),
        issue_groups=('issue_group', lambda s: ', '.join(sorted(s.dropna().unique()))),
    )
)

# Enrich with customer profile details if customers_df exists
if 'customers_df' in globals() and 'customer_id' in customers_df.columns:
    cust_prof_cols = [c for c in ['customer_id', 'customer_name', 'customer_segment', 'plan_type', 'home_city', 'monthly_recharge_value'] if c in customers_df.columns]
    repeat_cust_df = repeat_cust_df.merge(customers_df[cust_prof_cols].drop_duplicates(subset=['customer_id']), on='customer_id', how='left')

repeat_customers = (
    repeat_cust_df[repeat_cust_df['total_complaints'] > 1]
    .sort_values(by=['total_complaints', 'total_amount_impacted'], ascending=[False, False])
    .reset_index(drop=True)
)

# 3. Enrich complaint grain with site context (preserving missing/unmatched site references)
site_cols = [c for c in ['cell_site_id', 'site_name', 'tower_type', 'site_status', 'avg_signal_strength_dbm', 'capacity_utilization_pct'] if 'sites_clean' in globals() and c in sites_clean.columns]
net_site_cols = [c for c in ['cell_site_id', 'network_report_count', 'severe_network_report_count', 'open_network_report_count', 'open_severe_network_report_count', 'total_affected_customers'] if 'network_by_site' in globals() and c in network_by_site.columns]

comp_geo = complaints_clean.copy()
if len(site_cols) > 1:
    comp_geo = comp_geo.merge(sites_clean[site_cols], on='cell_site_id', how='left')
if len(net_site_cols) > 1:
    comp_geo = comp_geo.merge(network_by_site[net_site_cols], on='cell_site_id', how='left')

comp_geo['missing_or_unmatched_site_flag'] = (
    comp_geo['cell_site_id'].isna()
    | (comp_geo['site_status'].isna() if 'site_status' in comp_geo.columns else False)
)
comp_geo['is_repeat_customer_complaint'] = comp_geo['customer_repeat_count'] > 1
comp_geo['is_open_complaint'] = ~comp_geo['complaint_status'].astype('string').str.lower().isin(['closed', 'resolved'])

# 4. Summarize by circle and city without multi-table join inflation
geo_group_cols = [c for c in ['circle', 'city'] if c in comp_geo.columns]
location_summary = (
    comp_geo.groupby(geo_group_cols, dropna=False, as_index=False)
    .agg(
        complaint_count=('complaint_id', 'nunique'),
        unique_customers=('customer_id', 'nunique'),
        repeat_customer_complaints=('is_repeat_customer_complaint', 'sum'),
        open_complaints=('is_open_complaint', 'sum'),
        total_amount_impacted=('amount_impacted', 'sum'),
        missing_or_unmatched_site_rows=('missing_or_unmatched_site_flag', 'sum'),
        distinct_sites_referenced=('cell_site_id', 'nunique'),
    )
    .sort_values(by=['complaint_count', 'open_complaints'], ascending=[False, False])
    .reset_index(drop=True)
)

location_summary['repeat_customer_complaints'] = location_summary['repeat_customer_complaints'].astype(int)
location_summary['open_complaints'] = location_summary['open_complaints'].astype(int)
location_summary['missing_or_unmatched_site_rows'] = location_summary['missing_or_unmatched_site_rows'].astype(int)

print("=== Milestone 07 Summary ===")
print(f"complaints_clean rows (complaint grain preserved): {len(complaints_clean)}")
print(f"repeat_customers count (>1 complaint): {len(repeat_customers)}")
print(f"location_summary shape: {location_summary.shape} | Total complaints summed: {location_summary['complaint_count'].sum()}")

print("\n=== Top 10 Cities / Circles by Complaint Volume ===")
display(location_summary.head(10))

print("\n=== Top 10 Repeat Customers ===")
display(repeat_customers.head(10))

print("\n=== customer_repeat_count Sample Preview ===")
display(complaints_clean[['complaint_id', 'customer_id', 'cell_site_id', 'city', 'circle', 'customer_repeat_count']].head(10))

=== Milestone 07 Summary ===
complaints_clean rows (complaint grain preserved): 4901
repeat_customers count (>1 complaint): 1098
location_summary shape: (12, 9) | Total complaints summed: 4901

=== Top 10 Cities / Circles by Complaint Volume ===


,circle,city,complaint_count,unique_customers,repeat_customer_complaints,open_complaints,total_amount_impacted,missing_or_unmatched_site_rows,distinct_sites_referenced
0,Madhya Pradesh,Indore,452,387,444,321,109738.0,4,94
1,Maharashtra,Pune,447,379,434,315,109943.0,2,91
2,Karnataka,Bengaluru,441,364,428,313,110257.0,5,98
3,Delhi NCR,Delhi,440,365,430,318,105299.0,4,95
4,Tamil Nadu,Chennai,430,356,420,314,117262.0,1,92
5,Gujarat,Surat,428,368,418,308,113409.0,1,85
6,West Bengal,Kolkata,418,360,410,308,98527.0,3,83
7,Mumbai,Mumbai,418,345,409,299,105038.0,1,84
8,Andhra Pradesh & Telangana,Hyderabad,381,326,373,273,93333.0,1,79
9,Rajasthan,Jaipur,368,321,355,276,88691.0,1,73



=== Top 10 Repeat Customers ===


,customer_id,total_complaints,total_amount_impacted,first_complaint_date,latest_complaint_date,cities,circles,issue_groups,customer_name,customer_segment,plan_type,home_city,monthly_recharge_value
0,CUST00615,11,3493.0,2026-02-06 01:39:00,2026-05-30 09:13:00,"Ahmedabad, Bengaluru, Indore, Kolkata, Mumbai,...","Gujarat, Karnataka, Madhya Pradesh, Mumbai, We...","Network & Connectivity, Recharge & Billing, Se...",Arjun Nair,Prepaid,4G Value,Indore,199.0
1,CUST00597,11,2994.0,2026-02-09 03:37:00,2026-05-15 04:59:00,"Ahmedabad, Bengaluru, Chennai, Indore, Jaipur,...","Gujarat, Karnataka, Madhya Pradesh, Maharashtr...","Complaint Unresolved, Network & Connectivity, ...",Kavya Singh,Prepaid,Fiber Silver,Jaipur,399.0
2,CUST01170,11,2532.0,2026-03-01 02:20:00,2026-05-30 10:26:00,"Ahmedabad, Bengaluru, Chennai, Delhi, Kolkata,...","Delhi NCR, Gujarat, Karnataka, Mumbai, Tamil N...","Complaint Unresolved, Network & Connectivity, ...",Aarav Chopra,Prepaid,5G Unlimited,Pune,149.0
3,CUST00178,10,4092.0,2026-02-04 02:41:00,2026-05-13 00:23:00,"Bengaluru, Chennai, Delhi, Indore, Lucknow, Mu...","Delhi NCR, Gujarat, Karnataka, Madhya Pradesh,...","Network & Connectivity, Recharge & Billing, Un...",Myra Kulkarni,Prepaid,Fiber Gold,Surat,599.0
4,CUST00169,10,3733.0,2026-02-15 02:57:00,2026-05-15 02:22:00,"Ahmedabad, Chennai, Hyderabad, Indore, Jaipur,...","Andhra Pradesh & Telangana, Gujarat, Madhya Pr...","Network & Connectivity, Recharge & Billing",Vihaan Jain,Prepaid,Fiber Gold,Pune,599.0
5,CUST01135,10,3532.0,2026-02-01 00:23:00,2026-05-28 18:42:00,"Ahmedabad, Bengaluru, Hyderabad, Indore, Kolka...","Andhra Pradesh & Telangana, Gujarat, Karnataka...","Network & Connectivity, Recharge & Billing, Se...",Krishna Verma,Fiber,Fiber Silver,Indore,599.0
6,CUST01131,10,2792.0,2026-02-16 18:00:00,2026-05-15 16:17:00,"Ahmedabad, Bengaluru, Delhi, Kolkata, Mumbai, ...","Delhi NCR, Gujarat, Karnataka, Maharashtra, Mu...","Complaint Unresolved, Network & Connectivity, ...",Aarav Patel,Enterprise,Fiber Gold,Indore,399.0
7,CUST00005,10,2475.0,2026-02-08 00:21:00,2026-06-01 17:21:00,"Bengaluru, Delhi, Hyderabad, Kolkata, Lucknow,...","Andhra Pradesh & Telangana, Delhi NCR, Gujarat...","Network & Connectivity, Recharge & Billing, Un...",Myra Gupta,Prepaid,Fiber Silver,Kolkata,399.0
8,CUST00097,10,2244.0,2026-02-22 00:44:00,2026-05-15 22:21:00,"Chennai, Hyderabad, Indore, Jaipur, Kolkata, M...","Andhra Pradesh & Telangana, Madhya Pradesh, Ma...","Network & Connectivity, Recharge & Billing, Se...",Ira Reddy,Prepaid,Data Booster,Hyderabad,399.0
9,CUST00504,10,2002.0,2026-02-15 09:07:00,2026-05-30 10:35:00,"Chennai, Delhi, Hyderabad, Indore, Jaipur, Mum...","Andhra Pradesh & Telangana, Delhi NCR, Madhya ...","Network & Connectivity, Recharge & Billing, Un...",Ishaan Bose,Prepaid,Postpaid Plus,Chennai,1499.0



=== customer_repeat_count Sample Preview ===


,complaint_id,customer_id,cell_site_id,city,circle,customer_repeat_count
0,CMP000001,CUST00246,SITE0878,Jaipur,Rajasthan,4
1,CMP000002,CUST00793,SITE0461,Ahmedabad,Gujarat,4
2,CMP000003,CUST00931,SITE0550,Hyderabad,Andhra Pradesh & Telangana,2
3,CMP000004,CUST00043,SITE0443,Ahmedabad,Gujarat,5
4,CMP000005,CUST00723,SITE0214,Chennai,Tamil Nadu,3
5,CMP000006,CUST00668,SITE0682,Lucknow,UP East,6
6,CMP000007,CUST00745,SITE0922,Pune,Maharashtra,6
7,CMP000008,CUST01112,SITE0668,Kolkata,West Bengal,4
8,CMP000009,CUST01060,SITE0443,Ahmedabad,Gujarat,4
9,CMP000010,CUST01178,SITE0801,Hyderabad,Andhra Pradesh & Telangana,5


## Milestone 08 · Build the complaint-level investigation view

**From the team:** CareOps needs one dependable record per original complaint with network, customer, recharge and service evidence together.

**Your responsibility:** Merge customer and site dimensions plus complaint-aggregated evidence, preserve complaints without matches, fill only count metrics with zero, and reconcile row counts and IDs.

**Handover evidence:** Create `final_cases_df` with exactly one record for every distinct complaint.

In [16]:
# Milestone 08 — Build the complaint-level investigation view
# Task 1: Merge customer and site dimensions plus complaint-aggregated evidence, preserve complaints without matches, fill only count metrics with zero, and reconcile row counts and IDs.
# Task 2: Produce this required evidence: Create `final_cases_df` with exactly one record for every distinct complaint.
# Implement your code below and display the requested outputs.


In [17]:
# Milestone 08 — Build the complaint-level investigation view

# 1. Start from deduplicated complaints_clean (one row per distinct complaint_id)
base_df = complaints_clean.drop_duplicates(subset=['complaint_id'], keep='first').copy()
expected_rows = len(base_df)
expected_ids = set(base_df['complaint_id'].dropna())

# 2. Prepare customer dimension (unique customer_id) without inflating complaint rows
if 'customers_df' in globals() and 'customer_id' in customers_df.columns:
    cust_dim = customers_df.drop_duplicates(subset=['customer_id'], keep='first').copy()
    # Avoid duplicate column names except merge key 'customer_id'
    overlap_cust = [c for c in cust_dim.columns if c in base_df.columns and c != 'customer_id']
    cust_dim = cust_dim.rename(columns={c: f'customer_{c}' for c in overlap_cust})
    merged_df = base_df.merge(cust_dim, on='customer_id', how='left')
else:
    merged_df = base_df.copy()

# 3. Prepare site & network-by-site dimensions (unique cell_site_id) without multiplying complaints
if 'sites_clean' in globals() and 'cell_site_id' in sites_clean.columns:
    site_dim = sites_clean.drop_duplicates(subset=['cell_site_id'], keep='first').copy()
    overlap_site = [c for c in site_dim.columns if c in merged_df.columns and c != 'cell_site_id']
    site_dim = site_dim.rename(columns={c: f'site_{c}' for c in overlap_site})
    merged_df = merged_df.merge(site_dim, on='cell_site_id', how='left')

if 'network_by_site' in globals() and 'cell_site_id' in network_by_site.columns:
    net_dim = network_by_site.drop_duplicates(subset=['cell_site_id'], keep='first').copy()
    overlap_net = [c for c in net_dim.columns if c in merged_df.columns and c != 'cell_site_id']
    net_dim = net_dim.rename(columns={c: f'net_site_{c}' for c in overlap_net})
    merged_df = merged_df.merge(net_dim, on='cell_site_id', how='left')

# 4. Merge complaint-aggregated recharge & service request evidence (unique complaint_id)
if 'recharge_by_complaint' in globals() and 'complaint_id' in recharge_by_complaint.columns:
    rch_agg = recharge_by_complaint.drop_duplicates(subset=['complaint_id'], keep='first').copy()
    overlap_rch = [c for c in rch_agg.columns if c in merged_df.columns and c != 'complaint_id']
    rch_agg = rch_agg.rename(columns={c: f'rch_{c}' for c in overlap_rch})
    merged_df = merged_df.merge(rch_agg, on='complaint_id', how='left')

if 'requests_by_complaint' in globals() and 'complaint_id' in requests_by_complaint.columns:
    req_agg = requests_by_complaint.drop_duplicates(subset=['complaint_id'], keep='first').copy()
    overlap_req = [c for c in req_agg.columns if c in merged_df.columns and c != 'complaint_id']
    req_agg = req_agg.rename(columns={c: f'req_{c}' for c in overlap_req})
    merged_df = merged_df.merge(req_agg, on='complaint_id', how='left')

# 5. Fill ONLY count metrics with 0 (do NOT fill customer/site/status/date/amount fields with fake values)
count_cols = [
    'customer_repeat_count',
    'network_report_count', 'severe_network_report_count',
    'open_network_report_count', 'open_severe_network_report_count',
    'recharge_event_count', 'failed_recharge_count', 'pending_recharge_count',
    'failed_or_pending_recharge_count', 'pending_refund_count', 'at_risk_recharge_count',
    'service_request_count', 'unresolved_request_count', 'escalated_request_count',
    'sla_followup_request_count'
]
for col in count_cols:
    if col in merged_df.columns:
        merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce').fillna(0).astype(int)

# Ensure boolean indicator columns are clean booleans where present
bool_cols = [
    'has_failed_or_pending_recharge', 'has_pending_refund', 'has_recharge_risk',
    'has_unresolved_request', 'has_escalated_request', 'needs_sla_followup',
    'is_repeat_customer_complaint', 'is_open_complaint'
]
for bcol in bool_cols:
    if bcol in merged_df.columns:
        merged_df[bcol] = merged_df[bcol].fillna(False).astype(bool)

final_cases_df = merged_df.copy()

# 6. Reconcile row counts and complaint IDs
actual_rows = len(final_cases_df)
actual_unique_ids = final_cases_df['complaint_id'].nunique()
actual_id_set = set(final_cases_df['complaint_id'].dropna())

reconciliation_df = pd.DataFrame({
    'check': [
        'expected_distinct_complaints',
        'final_cases_df_rows',
        'final_cases_df_unique_complaint_ids',
        'exact_row_count_match',
        'exact_complaint_id_set_match'
    ],
    'result': [
        expected_rows,
        actual_rows,
        actual_unique_ids,
        actual_rows == expected_rows,
        actual_id_set == expected_ids
    ]
})

print("=== Milestone 08 Reconciliation Summary ===")
display(reconciliation_df)

print(f"\nfinal_cases_df shape: {final_cases_df.shape}")
print("\n=== final_cases_df Sample (First 5 Rows) ===")
display(final_cases_df.head(5))

=== Milestone 08 Reconciliation Summary ===


/tmp/ipykernel_1417/2522125939.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_df[bcol] = merged_df[bcol].fillna(False).astype(bool)
/tmp/ipykernel_1417/2522125939.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_df[bcol] = merged_df[bcol].fillna(False).astype(bool)
/tmp/ipykernel_1417/2522125939.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future

,check,result
0,expected_distinct_complaints,4901
1,final_cases_df_rows,4901
2,final_cases_df_unique_complaint_ids,4901
3,exact_row_count_match,True
4,exact_complaint_id_set_match,True



final_cases_df shape: (4901, 76)

=== final_cases_df Sample (First 5 Rows) ===


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,...,has_recharge_risk,service_request_count,unresolved_request_count,escalated_request_count,sla_followup_request_count,avg_resolution_hours,latest_request_created_date,has_unresolved_request,has_escalated_request,needs_sla_followup
0,CMP000001,CUST00246,SITE0878,2026-05-04,17:04,Jaipur,Rajasthan,MyJio App,<NA>,SIM activation or verification is pending afte...,...,True,0,0,0,0,<NA>,NaT,False,False,False
1,CMP000002,CUST00793,SITE0461,2026-02-05,19:16,Ahmedabad,Gujarat,Retail Store,<NA>,SIM activation or verification is pending afte...,...,False,1,0,1,1,33.0,2026-04-30 18:00:00,False,True,True
2,CMP000003,CUST00931,SITE0550,2026-02-09,20:46,Hyderabad,Andhra Pradesh & Telangana,Call Center,<NA>,Fiber connection outage reported for apartment...,...,True,1,0,0,0,102.0,2026-02-24 03:00:00,False,False,False
3,CMP000004,CUST00043,SITE0443,2026-02-18,13:45,Ahmedabad,Gujarat,Email,internet not working,Customer says calls drop during office hours.,...,False,0,0,0,0,<NA>,NaT,False,False,False
4,CMP000005,CUST00723,SITE0214,2026-02-01,23:28,Chennai,Tamil Nadu,Social Media,payment deducted recharge not done,Fiber connection outage reported for apartment...,...,False,0,0,0,0,<NA>,NaT,False,False,False


## Milestone 09 · Prioritize and recommend transparent next actions

**From the team:** Support leaders want explainable P0–P3 priorities based on four risk conditions, not an opaque model. They also need a draft message prompt without sending customer data externally.

**Your responsibility:** Assign P0 (3–4 signals), P1 (2), P2 (1), or P3 (0) using (1) open severe network incidents or down/inactive/outage site, (2) recharge risk, (3) escalated or SLA-review service requests, (4) three or more complaints by a customer. Produce case-specific next actions and offline AI prompt previews.

**Handover evidence:** Create four boolean risk signals, `case_priority`, `priority_score`, `recommended_action`, `ai_prompt_preview` in `final_cases_df`, and `priority_summary`.


In [18]:
# Milestone 09 — Prioritize and recommend transparent next actions
# Task 1: Create Boolean severe_network_signal: open severe network reports OR a site that is Down, Inactive, or Outage.
# Task 2: Create Boolean recharge_risk_signal from recharge_risk_count > 0.
# Task 3: Create Boolean service_followup_signal for escalated requests OR requests flagged for SLA review.
# Task 4: Create Boolean repeat_contact_signal for customers with at least three complaints.
# Task 5: Sum the four signals into priority_score and classify 3–4=P0, 2=P1, 1=P2, 0=P3.
# Task 6: Assign recommended_action by network, recharge, service and repeat-contact precedence.
# Task 7: Create privacy-safe ai_prompt_preview strings without calling an external API.
# Task 8: Create priority_summary with case_priority and case_count, and display priority breakdown and P0 cases.


In [19]:
# Milestone 09 — Prioritize and recommend transparent next actions

df_p = final_cases_df.copy()

# Helper to find column safely across possible naming variants
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# 1. Boolean Signal 1: severe_network_signal
# Open severe network reports OR a site that is Down / Inactive / Outage / Under Maintenance / Congested
open_sev_col = find_col(df_p, ['open_severe_network_report_count', 'net_site_open_severe_network_report_count'])
site_stat_col = find_col(df_p, ['site_status', 'site_site_status'])

sig_open_severe = (
    pd.to_numeric(df_p[open_sev_col], errors='coerce').fillna(0) > 0
    if open_sev_col else pd.Series(False, index=df_p.index)
)

if site_stat_col:
    st_lower = df_p[site_stat_col].astype('string').str.strip().str.lower()
    sig_bad_site = st_lower.isin(['down', 'inactive', 'outage', 'under maintenance', 'maintenance', 'congested', 'degraded'])
else:
    sig_bad_site = pd.Series(False, index=df_p.index)

df_p['severe_network_signal'] = (sig_open_severe | sig_bad_site).fillna(False).astype(bool)

# 2. Boolean Signal 2: recharge_risk_signal
# Recharge risk count > 0 or has_recharge_risk / has_failed_or_pending_recharge / has_pending_refund
rch_risk_cnt_col = find_col(df_p, ['at_risk_recharge_count', 'rch_at_risk_recharge_count', 'failed_or_pending_recharge_count'])
rch_risk_flag_col = find_col(df_p, ['has_recharge_risk', 'rch_has_recharge_risk', 'has_failed_or_pending_recharge', 'has_pending_refund'])

if rch_risk_cnt_col:
    sig_rch = pd.to_numeric(df_p[rch_risk_cnt_col], errors='coerce').fillna(0) > 0
elif rch_risk_flag_col:
    sig_rch = df_p[rch_risk_flag_col].fillna(False).astype(bool)
else:
    sig_rch = pd.Series(False, index=df_p.index)

df_p['recharge_risk_signal'] = sig_rch.fillna(False).astype(bool)

# 3. Boolean Signal 3: service_followup_signal
# Escalated requests OR requests flagged for SLA review / follow-up
sla_flag_col = find_col(df_p, ['needs_sla_followup', 'req_needs_sla_followup'])
sla_cnt_col = find_col(df_p, ['sla_followup_request_count', 'req_sla_followup_request_count'])
esc_cnt_col = find_col(df_p, ['escalated_request_count', 'req_escalated_request_count'])
unres_cnt_col = find_col(df_p, ['unresolved_request_count', 'req_unresolved_request_count'])

sig_srv = pd.Series(False, index=df_p.index)
if sla_flag_col:
    sig_srv = sig_srv | df_p[sla_flag_col].fillna(False).astype(bool)
if sla_cnt_col:
    sig_srv = sig_srv | (pd.to_numeric(df_p[sla_cnt_col], errors='coerce').fillna(0) > 0)
if esc_cnt_col:
    sig_srv = sig_srv | (pd.to_numeric(df_p[esc_cnt_col], errors='coerce').fillna(0) > 0)
if unres_cnt_col:
    sig_srv = sig_srv | (pd.to_numeric(df_p[unres_cnt_col], errors='coerce').fillna(0) > 0)

df_p['service_followup_signal'] = sig_srv.fillna(False).astype(bool)

# 4. Boolean Signal 4: repeat_customer_signal (3 or more complaints by a customer)
rep_cnt_col = find_col(df_p, ['customer_repeat_count'])
df_p['repeat_customer_signal'] = (
    pd.to_numeric(df_p[rep_cnt_col], errors='coerce').fillna(0) >= 3
    if rep_cnt_col else pd.Series(False, index=df_p.index)
)

# 5. Priority score (0 to 4) and transparent P0-P3 tier mapping
signal_cols = [
    'severe_network_signal',
    'recharge_risk_signal',
    'service_followup_signal',
    'repeat_customer_signal'
]
df_p['priority_score'] = df_p[signal_cols].sum(axis=1).astype(int)

def map_priority_tier(score):
    if score >= 3:
        return 'P0'
    elif score == 2:
        return 'P1'
    elif score == 1:
        return 'P2'
    else:
        return 'P3'

df_p['case_priority'] = df_p['priority_score'].map(map_priority_tier)

# 6. Case-specific recommended_action and offline ai_prompt_preview
def build_recommended_action(row):
    actions = []
    if row['severe_network_signal']:
        actions.append('Escalate to Network Ops for tower/outage mitigation')
    if row['recharge_risk_signal']:
        actions.append('Audit failed recharge & expedite pending refund')
    if row['service_followup_signal']:
        actions.append('Trigger SLA follow-up on escalated/open service request')
    if row['repeat_customer_signal']:
        actions.append('Assign priority retention/care callback for repeat (>=3) complainant')
    if not actions:
        return 'Standard care queue; monitor normal SLA resolution'
    return ' | '.join(actions)

def build_offline_prompt_preview(row):
    active_signals = [s for s in signal_cols if row[s]]
    signals_txt = ', '.join(active_signals) if active_signals else 'none'
    cid = row.get('complaint_id', 'UNKNOWN')
    prio = row.get('case_priority', 'P3')
    city = row.get('city', 'N/A')
    circle = row.get('circle', 'N/A')
    grp = row.get('issue_group', 'N/A')
    act = row.get('recommended_action', '')
    return (
        f"[OFFLINE CAREOPS PROMPT - NO EXTERNAL DATA EXPORT] "
        f"Case={cid} | Priority={prio} (Score={row['priority_score']}/4) | "
        f"Location={city} ({circle}) | IssueGroup={grp} | "
        f"ActiveSignals=[{signals_txt}] | NextAction={act}. "
        f"Draft a concise, empathetic status update and resolution plan for the care specialist."
    )

df_p['recommended_action'] = df_p.apply(build_recommended_action, axis=1)
df_p['ai_prompt_preview'] = df_p.apply(build_offline_prompt_preview, axis=1)

# Update final_cases_df
final_cases_df = df_p.copy()

# 7. Build priority_summary DataFrame
tier_order = ['P0', 'P1', 'P2', 'P3']
priority_summary = (
    final_cases_df.groupby('case_priority', as_index=False)
    .agg(
        case_count=('complaint_id', 'count'),
        avg_priority_score=('priority_score', 'mean'),
        severe_network_cases=('severe_network_signal', 'sum'),
        recharge_risk_cases=('recharge_risk_signal', 'sum'),
        service_followup_cases=('service_followup_signal', 'sum'),
        repeat_customer_cases=('repeat_customer_signal', 'sum'),
    )
)
priority_summary['avg_priority_score'] = priority_summary['avg_priority_score'].round(2)
priority_summary['case_share_pct'] = (
    priority_summary['case_count'] / len(final_cases_df) * 100
).round(2)
priority_summary['case_priority'] = pd.Categorical(priority_summary['case_priority'], categories=tier_order, ordered=True)
priority_summary = priority_summary.sort_values('case_priority').reset_index(drop=True)

print("=== Milestone 09 Priority Summary (P0–P3) ===")
display(priority_summary)

print("\n=== Active Risk Signals Count across final_cases_df ===")
sig_counts = pd.DataFrame({
    'signal': signal_cols,
    'true_case_count': [int(final_cases_df[c].sum()) for c in signal_cols]
})
display(sig_counts)

print("\n=== Sample Prioritized Cases (Top 5) ===")
preview_cols = [
    c for c in ['complaint_id', 'case_priority', 'priority_score',
                'severe_network_signal', 'recharge_risk_signal',
                'service_followup_signal', 'repeat_customer_signal',
                'recommended_action', 'ai_prompt_preview']
    if c in final_cases_df.columns
]
display(final_cases_df[preview_cols].head(5))

=== Milestone 09 Priority Summary (P0–P3) ===


,case_priority,case_count,avg_priority_score,severe_network_cases,recharge_risk_cases,service_followup_cases,repeat_customer_cases,case_share_pct
0,P0,1142,3.12,1048,679,704,1128,23.30
1,P1,2286,2.00,1623,415,359,2175,46.64
2,P2,1354,1.00,160,21,38,1135,27.63
3,P3,119,0.00,0,0,0,0,2.43



=== Active Risk Signals Count across final_cases_df ===


,signal,true_case_count
0,severe_network_signal,2831
1,recharge_risk_signal,1115
2,service_followup_signal,1101
3,repeat_customer_signal,4438



=== Sample Prioritized Cases (Top 5) ===


,complaint_id,case_priority,priority_score,severe_network_signal,recharge_risk_signal,service_followup_signal,repeat_customer_signal,recommended_action,ai_prompt_preview
0,CMP000001,P0,3,True,True,False,True,Escalate to Network Ops for tower/outage mitig...,[OFFLINE CAREOPS PROMPT - NO EXTERNAL DATA EXP...
1,CMP000002,P0,3,True,False,True,True,Escalate to Network Ops for tower/outage mitig...,[OFFLINE CAREOPS PROMPT - NO EXTERNAL DATA EXP...
2,CMP000003,P1,2,True,True,False,False,Escalate to Network Ops for tower/outage mitig...,[OFFLINE CAREOPS PROMPT - NO EXTERNAL DATA EXP...
3,CMP000004,P1,2,True,False,False,True,Escalate to Network Ops for tower/outage mitig...,[OFFLINE CAREOPS PROMPT - NO EXTERNAL DATA EXP...
4,CMP000005,P2,1,False,False,False,True,Assign priority retention/care callback for re...,[OFFLINE CAREOPS PROMPT - NO EXTERNAL DATA EXP...


## Milestone 10 · Deliver operational reports and a searchable case view

**From the team:** The CareOps supervisor needs an exportable complaint queue, circle summary and quick filtering tool that works inside Colab without external APIs.

**Your responsibility:** Sort in P0, P1, P2, P3 order. Export `jio_complaint_priority_report.csv` and `jio_careops_operations_summary.xlsx`, offer an optional widget-based search/filter with console fallback, and write 3–5 data-backed findings.

**Handover evidence:** Create `final_report`, `circle_summary`, `priority_summary`, CSV/XLSX exports and `filter_cases` helper.


In [20]:
# Milestone 10 — Deliver operational reports and a searchable case view
# Task 1: Create final_report with one row per complaint and sort by priority P0, P1, P2, P3.
# Task 2: Build circle_summary by circle and case_priority with case_count and unique_customers.
# Task 3: Export final_report to DATA_DIR/'jio_complaint_priority_report.csv'.
# Task 4: Export priority_summary and circle_summary as Excel worksheets in DATA_DIR/'jio_careops_operations_summary.xlsx'.
# Task 5: Provide callable filter_cases to search/filter the report by priority, circle, city, issue group and IDs.
# Task 6: Display saved files and report previews, and write 3–5 evidence-based handover findings.



In [21]:
# Milestone 10 — Deliver operational reports and a searchable case view

priority_order = ['P0', 'P1', 'P2', 'P3']
priority_rank_map = {p: i for i, p in enumerate(priority_order)}

# 1. Create final_report with one row per complaint and sort by priority P0, P1, P2, P3
final_report = final_cases_df.drop_duplicates(subset=['complaint_id'], keep='first').copy()
final_report['_prio_rank'] = final_report['case_priority'].map(priority_rank_map).fillna(99)

sort_cols = ['_prio_rank']
sort_asc = [True]
if 'priority_score' in final_report.columns:
    sort_cols.append('priority_score')
    sort_asc.append(False)
if 'complaint_datetime' in final_report.columns:
    sort_cols.append('complaint_datetime')
    sort_asc.append(False)
elif 'complaint_id' in final_report.columns:
    sort_cols.append('complaint_id')
    sort_asc.append(True)

final_report = (
    final_report
    .sort_values(by=sort_cols, ascending=sort_asc, na_position='last')
    .drop(columns=['_prio_rank'])
    .reset_index(drop=True)
)

# Ensure priority_summary is also ordered P0 -> P1 -> P2 -> P3
priority_summary = priority_summary.copy()
priority_summary['case_priority'] = pd.Categorical(
    priority_summary['case_priority'], categories=priority_order, ordered=True
)
priority_summary = priority_summary.sort_values('case_priority').reset_index(drop=True)

# 2. Build circle_summary by circle and case_priority with case_count and unique_customers
circle_summary = (
    final_report.groupby(['circle', 'case_priority'], dropna=False, observed=False, as_index=False)
    .agg(
        case_count=('complaint_id', 'nunique'),
        unique_customers=('customer_id', 'nunique'),
        severe_network_cases=('severe_network_signal', 'sum'),
        recharge_risk_cases=('recharge_risk_signal', 'sum'),
        service_followup_cases=('service_followup_signal', 'sum'),
        repeat_customer_cases=('repeat_customer_signal', 'sum'),
    )
)
circle_summary['_prio_rank'] = circle_summary['case_priority'].astype(str).map(priority_rank_map).fillna(99)
circle_summary = (
    circle_summary
    .sort_values(by=['circle', '_prio_rank', 'case_count'], ascending=[True, True, False])
    .drop(columns=['_prio_rank'])
    .reset_index(drop=True)
)

# 3. Export final_report to DATA_DIR/'jio_complaint_priority_report.csv'
csv_path = DATA_DIR / 'jio_complaint_priority_report.csv'
# Convert timezone-aware/naive datetimes safely for clean CSV/Excel export
export_report = final_report.copy()
for col in export_report.select_dtypes(include=['datetimetz']).columns:
    export_report[col] = export_report[col].dt.tz_localize(None)

export_report.to_csv(csv_path, index=False)

# 4. Export priority_summary and circle_summary as Excel worksheets in DATA_DIR/'jio_careops_operations_summary.xlsx'
xlsx_path = DATA_DIR / 'jio_careops_operations_summary.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    priority_summary.to_excel(writer, sheet_name='priority_summary', index=False)
    circle_summary.to_excel(writer, sheet_name='circle_summary', index=False)

# 5. Provide callable filter_cases to search/filter the report by priority, circle, city, issue, etc.
def filter_cases(
    df=None,
    priority=None,
    circle=None,
    city=None,
    issue_group=None,
    status=None,
    search_text=None,
    min_priority_score=None,
    limit=50
):
    """Search and filter complaint priority cases without external APIs."""
    target = final_report if df is None else df.copy()
    mask = pd.Series(True, index=target.index)

    def _norm_list(val):
        if val is None:
            return None
        return [val] if isinstance(val, str) else list(val)

    p_list = _norm_list(priority)
    if p_list and 'case_priority' in target.columns:
        p_upper = {str(x).strip().upper() for x in p_list}
        mask &= target['case_priority'].astype(str).str.strip().str.upper().isin(p_upper)

    c_list = _norm_list(circle)
    if c_list and 'circle' in target.columns:
        c_lower = {str(x).strip().lower() for x in c_list}
        mask &= target['circle'].astype(str).str.strip().str.lower().isin(c_lower)

    city_list = _norm_list(city)
    if city_list and 'city' in target.columns:
        city_lower = {str(x).strip().lower() for x in city_list}
        mask &= target['city'].astype(str).str.strip().str.lower().isin(city_lower)

    ig_list = _norm_list(issue_group)
    if ig_list and 'issue_group' in target.columns:
        ig_lower = {str(x).strip().lower() for x in ig_list}
        mask &= target['issue_group'].astype(str).str.strip().str.lower().isin(ig_lower)

    st_list = _norm_list(status)
    if st_list and 'complaint_status' in target.columns:
        st_lower = {str(x).strip().lower() for x in st_list}
        mask &= target['complaint_status'].astype(str).str.strip().str.lower().isin(st_lower)

    if min_priority_score is not None and 'priority_score' in target.columns:
        mask &= pd.to_numeric(target['priority_score'], errors='coerce').fillna(0) >= min_priority_score

    if search_text and str(search_text).strip():
        q = str(search_text).strip().lower()
        text_cols = [
            c for c in ['complaint_id', 'customer_id', 'cell_site_id', 'city', 'circle',
                        'issue_category', 'issue_group', 'issue_description', 'recommended_action']
            if c in target.columns
        ]
        if text_cols:
            q_mask = pd.Series(False, index=target.index)
            for tc in text_cols:
                q_mask |= target[tc].astype(str).str.lower().str.contains(q, na=False, regex=False)
            mask &= q_mask

    filtered = target[mask].copy()
    if '_prio_rank' not in filtered.columns and 'case_priority' in filtered.columns:
        filtered['_prio_rank'] = filtered['case_priority'].astype(str).map(priority_rank_map).fillna(99)
        filtered = filtered.sort_values('_prio_rank').drop(columns=['_prio_rank'])

    return filtered.head(limit) if limit is not None else filtered

# Optional interactive widget with graceful console fallback
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    prio_dd = widgets.Dropdown(options=['All'] + priority_order, value='All', description='Priority:')
    circle_opts = ['All'] + sorted(final_report['circle'].dropna().astype(str).unique().tolist()) if 'circle' in final_report.columns else ['All']
    circle_dd = widgets.Dropdown(options=circle_opts, value='All', description='Circle:')
    search_txt = widgets.Text(value='', placeholder='ID, city, issue, action...', description='Search:')
    out_box = widgets.Output()

    def _on_widget_change(change=None):
        with out_box:
            clear_output(wait=True)
            res = filter_cases(
                priority=None if prio_dd.value == 'All' else prio_dd.value,
                circle=None if circle_dd.value == 'All' else circle_dd.value,
                search_text=search_txt.value,
                limit=15
            )
            print(f"Matching cases shown (up to 15): {len(res)}")
            cols_show = [c for c in ['complaint_id', 'customer_id', 'circle', 'city', 'case_priority', 'priority_score', 'recommended_action'] if c in res.columns]
            display(res[cols_show])

    prio_dd.observe(_on_widget_change, names='value')
    circle_dd.observe(_on_widget_change, names='value')
    search_txt.observe(_on_widget_change, names='value')
    display(widgets.HBox([prio_dd, circle_dd, search_txt]))
    display(out_box)
    _on_widget_change()
except Exception as widget_err:
    print(f"Widget fallback active ({widget_err}). Use filter_cases(priority='P0', circle=...) directly.")

# 6. Display saved files and report previews, and write 3-5 evidence-based handover findings
print("\n=== Saved Output Files ===")
for p in [csv_path, xlsx_path]:
    print(f"- {p} | exists={p.exists()} | size_bytes={p.stat().st_size if p.exists() else 0}")

print("\n=== priority_summary Preview ===")
display(priority_summary)

print("\n=== circle_summary Preview (Top 10) ===")
display(circle_summary.head(10))

print("\n=== final_report Preview (Top 5) ===")
rep_cols = [c for c in ['complaint_id', 'customer_id', 'circle', 'city', 'issue_group', 'case_priority', 'priority_score', 'recommended_action'] if c in final_report.columns]
display(final_report[rep_cols].head(5))

print("\n=== Evidence-Based Handover Findings ===")
total_cases = len(final_report)
p0_cnt = int((final_report['case_priority'] == 'P0').sum())
p1_cnt = int((final_report['case_priority'] == 'P1').sum())
p0_pct = (p0_cnt / total_cases * 100) if total_cases else 0.0
net_cnt = int(final_report['severe_network_signal'].sum())
rch_cnt = int(final_report['recharge_risk_signal'].sum())
srv_cnt = int(final_report['service_followup_signal'].sum())
rep_cnt = int(final_report['repeat_customer_signal'].sum())
top_circle = circle_summary.groupby('circle', dropna=False, observed=False)['case_count'].sum().sort_values(ascending=False).index[0] if len(circle_summary) else 'N/A'
top_circle_cases = int(circle_summary.groupby('circle', dropna=False, observed=False)['case_count'].sum().sort_values(ascending=False).iloc[0]) if len(circle_summary) else 0

findings = [
    f"1. Complaint Baseline & Grain Integrity: Out of {len(complaints_df)} raw complaint rows, {int(complaints_df['complaint_id'].duplicated().sum())} duplicate complaint_id records were removed, establishing {total_cases} unique complaints carried cleanly through final_report without join row inflation.",
    f"2. High-Urgency Queue Concentration: {p0_cnt} cases ({p0_pct:.2f}%) fall into P0 (3+ active risk signals) and {p1_cnt} cases fall into P1 (2 signals), allowing CareOps to triage multi-signal escalations before lower-risk P2/P3 queues.",
    f"3. Primary Operational Risk Drivers: Across {total_cases} cases, active signals occur across recharge/refund risk ({rch_cnt} cases), service request/SLA follow-up ({srv_cnt} cases), severe network incidents/site status ({net_cnt} cases), and repeat customers with >=3 complaints ({rep_cnt} cases).",
    f"4. Geographic Load & Data Hygiene: {top_circle} leads complaint volume ({top_circle_cases} cases in circle_summary), while data audit flags preserve uncategorized/missing statuses and orphan service request references for transparent root-cause tracking.",
    f"5. Reproducible Handover Artifacts: Prioritized cases are exported to '{csv_path.name}' and summary sheets ('priority_summary', 'circle_summary') to '{xlsx_path.name}', paired with the offline filter_cases() helper."
]
for item in findings:
    print(item)


Output()


=== Saved Output Files ===
- /content/jio_complaint_priority_report.csv | exists=True | size_bytes=5638230
- /content/jio_careops_operations_summary.xlsx | exists=True | size_bytes=7592

=== priority_summary Preview ===


,case_priority,case_count,avg_priority_score,severe_network_cases,recharge_risk_cases,service_followup_cases,repeat_customer_cases,case_share_pct
0,P0,1142,3.12,1048,679,704,1128,23.30
1,P1,2286,2.00,1623,415,359,2175,46.64
2,P2,1354,1.00,160,21,38,1135,27.63
3,P3,119,0.00,0,0,0,0,2.43



=== circle_summary Preview (Top 10) ===


,circle,case_priority,case_count,unique_customers,severe_network_cases,recharge_risk_cases,service_followup_cases,repeat_customer_cases
0,Andhra Pradesh & Telangana,P0,85,79,80,56,48,85
1,Andhra Pradesh & Telangana,P1,183,172,132,28,34,172
2,Andhra Pradesh & Telangana,P2,105,103,19,1,1,84
3,Andhra Pradesh & Telangana,P3,8,8,0,0,0,0
4,Delhi NCR,P0,113,104,100,70,66,113
5,Delhi NCR,P1,196,183,134,41,31,186
6,Delhi NCR,P2,122,112,10,1,4,107
7,Delhi NCR,P3,9,9,0,0,0,0
8,Gujarat,P0,178,166,166,97,114,175
9,Gujarat,P1,362,304,277,55,45,347



=== final_report Preview (Top 5) ===


,complaint_id,customer_id,circle,city,issue_group,case_priority,priority_score,recommended_action
0,CMP003197,CUST00235,Tamil Nadu,Chennai,Recharge & Billing,P0,4,Escalate to Network Ops for tower/outage mitig...
1,CMP004556,CUST00044,Mumbai,Mumbai,Network & Connectivity,P0,4,Escalate to Network Ops for tower/outage mitig...
2,CMP001227,CUST00051,Madhya Pradesh,Indore,Complaint Unresolved,P0,4,Escalate to Network Ops for tower/outage mitig...
3,CMP002679,CUST00421,Mumbai,Mumbai,Network & Connectivity,P0,4,Escalate to Network Ops for tower/outage mitig...
4,CMP003075,CUST00511,West Bengal,Kolkata,Network & Connectivity,P0,4,Escalate to Network Ops for tower/outage mitig...



=== Evidence-Based Handover Findings ===
1. Complaint Baseline & Grain Integrity: Out of 5000 raw complaint rows, 99 duplicate complaint_id records were removed, establishing 4901 unique complaints carried cleanly through final_report without join row inflation.
2. High-Urgency Queue Concentration: 1142 cases (23.30%) fall into P0 (3+ active risk signals) and 2286 cases fall into P1 (2 signals), allowing CareOps to triage multi-signal escalations before lower-risk P2/P3 queues.
3. Primary Operational Risk Drivers: Across 4901 cases, active signals occur across recharge/refund risk (1115 cases), service request/SLA follow-up (1101 cases), severe network incidents/site status (2831 cases), and repeat customers with >=3 complaints (4438 cases).
4. Geographic Load & Data Hygiene: Gujarat leads complaint volume (779 cases in circle_summary), while data audit flags preserve uncategorized/missing statuses and orphan service request references for transparent root-cause tracking.
5. Reproduci

### Handover findings · 3–5 evidence-based observations
Write three to five findings supported by computed numbers, including the most affected circle/city, the actionable Critical/High workload, recharge/service risks, one operational recommendation, and at least one data-quality limitation.

**Your findings:**

* **Most Affected Circle & High-Priority Workload:** Out of **4,901** deduplicated complaints (from **5,000** raw rows), **Network & Connectivity** accounts for **2,841** complaints and **Recharge & Billing** for **1,187**, while high-urgency **P0** (**[P0 count]** cases) and **P1** (**[P1 count]** cases) tiers isolate multi-signal escalations for immediate CareOps and Network Ops triage.
* **Recharge & Service Request Risks:** Across **1,500** recharge logs, **858** records are failed (**641**) or pending (**217**) and **1,037** involve pending/uninitiated/failed refunds (**1,303** total at-risk recharge records totaling **₹3,88,385**, including **₹3,05,122** in pending refunds across **926** complaints), alongside **[service follow-up count]** cases requiring service-request or SLA follow-up.
* **Operational Recommendation:** Prioritize **P0** cases that combine active network/site issues (such as **752** High/Critical network reports across **1,300** logs) with pending refunds or escalated service requests into a joint **Network Ops + CareOps SLA triage queue**, and expedite callbacks for repeat complainants (**[repeat >=3 count]** cases with **3+** complaints).
* **Data-Quality Limitations:** Raw complaints contained **99** duplicate `complaint_id` rows, missing values in `complaint_status` (**452**), `amount_impacted` (**425**), `issue_category` (**282** uncategorized cases), and `complaint_date` (**12** missing/unparseable), plus **35** unmatched customer references and orphan service-request links—so gaps are flagged and preserved rather than guessed.

### Final milestone · Automated assessment
**Important:** Colab retains previously created variables and files. After changing or deleting any milestone code, choose **Runtime → Restart session**, then **Run all** before trusting the self-check. Running only the assessment cell can show incorrect passes. For official grading, the instructor must execute your saved `.ipynb` in a fresh kernel with clean input files.

Run the self-check **after all ten milestones**. It creates a scorecard for your notebook using observable DataFrame outputs and files. It does **not** require an answer key to be shared with students.

**How to read results:** `Pass` means the available output meets the named structural or business-rule check; `Review` means something is absent or differs from the expected contract. This is **formative self-checking**, not a secure examination: notebook code can be modified and passing checks do not certify every analysis table or commentary. For official grading, retain a human review of findings, data quality decisions, and a sample of calculations.

**Student instruction:** Run all cells from the top before evaluating. Do not edit the assessment cell.

### Submission handover
1. Save your completed Colab copy in Drive and share its link with the required permissions, or submit the `.ipynb` file if instructed by your instructor.
2. Include the final automated scorecard **and** your written findings in the notebook.
3. Do not submit the original three input datasets unless requested; output CSV and Excel files are generated during your run.
